# Two-dimensional box filter with shared memory

This notebook compares a direct 2D box filter with a tiled implementation that loads each block's image region and halo into shared memory. Both kernels compute the same zero-padded convolution and are checked against a CPU reference. A small non-divisible image makes the boundary and partial-block behavior visible; larger images are used for kernel-only timing.

## Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU**. This cell displays the GPU and CUDA compiler used for the measurements.

In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sun Nov 15 12:55:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA error and timing support

`CUDA_CHECK` handles CUDA errors independently of `TimingGPU`. The timer uses CUDA events. The benchmark measures repeated kernel launches only; allocation, copies, CPU reference calculations, and result checks are outside the timed interval.

In [2]:
%%writefile code/cuda_check.cuh
#ifndef CUDA_CHECK_CUH
#define CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    cudaError_t status = (call); \
    if (status != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


Writing code/cuda_check.cuh


In [3]:
%%writefile code/TimingGPU.cuh
#ifndef TIMING_GPU_CUH
#define TIMING_GPU_CUH
#include <cuda_runtime.h>
class TimingGPU {
    cudaEvent_t start_;
    cudaEvent_t stop_;
    bool started_;
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    float GetCounter(); // milliseconds
};
#endif


Writing code/TimingGPU.cuh


In [4]:
%%writefile code/TimingGPU.cu
#include "TimingGPU.cuh"
#include "cuda_check.cuh"
#include <cstdio>
#include <cstdlib>

TimingGPU::TimingGPU() : started_(false) {
    CUDA_CHECK(cudaEventCreate(&start_));
    CUDA_CHECK(cudaEventCreate(&stop_));
}
TimingGPU::~TimingGPU() {
    CUDA_CHECK(cudaEventDestroy(start_));
    CUDA_CHECK(cudaEventDestroy(stop_));
}
void TimingGPU::StartCounter() {
    CUDA_CHECK(cudaEventRecord(start_));
    started_ = true;
}
float TimingGPU::GetCounter() {
    if (!started_) {
        std::fprintf(stderr, "Call StartCounter before GetCounter.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(stop_));
    CUDA_CHECK(cudaEventSynchronize(stop_));
    float milliseconds = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start_, stop_));
    started_ = false;
    return milliseconds;
}


Writing code/TimingGPU.cu


## Kernels and host program

The filter is a uniform `K×K` average with odd `K`. Samples outside the image are zero, and the divisor remains `K²` at the boundaries. The direct kernel checks each input coordinate. The shared kernel cooperatively fills a 2D tile of `(blockDim.x+2R) × (blockDim.y+2R)` values, including the halo, then all threads synchronize. Only valid image pixels are written, so threads in a partial edge block still load data and reach the barrier but do not write an output.

In [5]:
%%writefile code/chapter02_2d_box_filter.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include <cmath>
#include <cstdio>
#include <cstdlib>

const int BLOCK_X = 32;
const int BLOCK_Y = 8;
const int REPEATS = 20;
const int BATCHES = 5;

__global__ void box_filter_direct(const float *image, float *output,
                                  int height, int width, int k) {
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    const int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x < width && y < height) {
        const int radius = k / 2;
        float sum = 0.0f;
        for (int fy = -radius; fy <= radius; ++fy) {
            for (int fx = -radius; fx <= radius; ++fx) {
                const int input_x = x + fx;
                const int input_y = y + fy;
                if (input_x >= 0 && input_x < width &&
                    input_y >= 0 && input_y < height)
                    sum += image[input_y * width + input_x];
            }
        }
        output[y * width + x] = sum / float(k * k);
    }
}

__global__ void box_filter_shared(const float *image, float *output,
                                  int height, int width, int k) {
    extern __shared__ float tile[];
    const int radius = k / 2;
    const int tile_width = blockDim.x + 2 * radius;
    const int tile_height = blockDim.y + 2 * radius;
    const int block_start_x = blockIdx.x * blockDim.x;
    const int block_start_y = blockIdx.y * blockDim.y;

    // Cooperatively load the tile and its halo. Loads are contiguous in x.
    for (int tile_y = threadIdx.y; tile_y < tile_height; tile_y += blockDim.y) {
        for (int tile_x = threadIdx.x; tile_x < tile_width; tile_x += blockDim.x) {
            const int input_x = block_start_x + tile_x - radius;
            const int input_y = block_start_y + tile_y - radius;
            const int tile_index = tile_y * tile_width + tile_x;
            tile[tile_index] =
                (input_x >= 0 && input_x < width &&
                 input_y >= 0 && input_y < height)
                ? image[input_y * width + input_x] : 0.0f;
        }
    }
    __syncthreads();

    const int x = block_start_x + threadIdx.x;
    const int y = block_start_y + threadIdx.y;
    if (x < width && y < height) {
        float sum = 0.0f;
        for (int fy = 0; fy < k; ++fy) {
            for (int fx = 0; fx < k; ++fx) {
                const int tile_index = (threadIdx.y + fy) * tile_width
                                     + threadIdx.x + fx;
                sum += tile[tile_index];
            }
        }
        output[y * width + x] = sum / float(k * k);
    }
}

void launch_direct(const float *image, float *output,
                   int height, int width, int k) {
    const dim3 block(BLOCK_X, BLOCK_Y);
    const dim3 grid((width + BLOCK_X - 1) / BLOCK_X,
                    (height + BLOCK_Y - 1) / BLOCK_Y);
    box_filter_direct<<<grid, block>>>(image, output, height, width, k);
    CUDA_CHECK(cudaGetLastError());
}

void launch_shared(const float *image, float *output,
                   int height, int width, int k) {
    const dim3 block(BLOCK_X, BLOCK_Y);
    const dim3 grid((width + BLOCK_X - 1) / BLOCK_X,
                    (height + BLOCK_Y - 1) / BLOCK_Y);
    const int radius = k / 2;
    const size_t tile_width = size_t(BLOCK_X + 2 * radius);
    const size_t tile_height = size_t(BLOCK_Y + 2 * radius);
    const size_t shared_bytes = tile_width * tile_height * sizeof(float);
    box_filter_shared<<<grid, block, shared_bytes>>>(
        image, output, height, width, k);
    CUDA_CHECK(cudaGetLastError());
}

void make_image(float *image, int height, int width) {
    for (int y = 0; y < height; ++y)
        for (int x = 0; x < width; ++x)
            image[y * width + x] = float((7 * y + 3 * x) % 23) / 22.0f;
}

void make_cpu_reference(const float *image, float *reference,
                        int height, int width, int k) {
    const int radius = k / 2;
    for (int y = 0; y < height; ++y) {
        for (int x = 0; x < width; ++x) {
            float sum = 0.0f;
            for (int fy = -radius; fy <= radius; ++fy) {
                for (int fx = -radius; fx <= radius; ++fx) {
                    const int input_x = x + fx;
                    const int input_y = y + fy;
                    if (input_x >= 0 && input_x < width &&
                        input_y >= 0 && input_y < height)
                        sum += image[input_y * width + input_x];
                }
            }
            reference[y * width + x] = sum / float(k * k);
        }
    }
}

void check_result(const float *gpu, const float *reference,
                  int count, const char *name) {
    for (int i = 0; i < count; ++i) {
        const float tolerance = 3.0e-5f * (1.0f + std::fabs(reference[i]));
        if (std::fabs(gpu[i] - reference[i]) > tolerance) {
            std::fprintf(stderr,
                "%s mismatch at index %d: CPU=%.8f GPU=%.8f\n",
                name, i, reference[i], gpu[i]);
            std::exit(EXIT_FAILURE);
        }
    }
}

void run_case(int height, int width, int k,
              bool print_matrices, bool measure_time) {
    if (k <= 0 || (k % 2) == 0) {
        std::fprintf(stderr, "Filter width must be a positive odd number.\n");
        std::exit(EXIT_FAILURE);
    }
    const int count = height * width;
    const size_t bytes = size_t(count) * sizeof(float);
    float *h_image = (float*)std::malloc(bytes);
    float *h_reference = (float*)std::malloc(bytes);
    float *h_direct = (float*)std::malloc(bytes);
    float *h_shared = (float*)std::malloc(bytes);
    float *d_image = nullptr, *d_direct = nullptr, *d_shared = nullptr;
    if (!h_image || !h_reference || !h_direct || !h_shared) {
        std::fprintf(stderr, "Host allocation failed.\n");
        std::exit(EXIT_FAILURE);
    }
    make_image(h_image, height, width);
    make_cpu_reference(h_image, h_reference, height, width, k);

    CUDA_CHECK(cudaMalloc((void**)&d_image, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_direct, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_shared, bytes));
    CUDA_CHECK(cudaMemcpy(d_image, h_image, bytes, cudaMemcpyHostToDevice));

    launch_direct(d_image, d_direct, height, width, k);
    launch_shared(d_image, d_shared, height, width, k);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_direct, d_direct, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_shared, d_shared, bytes, cudaMemcpyDeviceToHost));
    check_result(h_direct, h_reference, count, "Direct kernel");
    check_result(h_shared, h_reference, count, "Shared-memory kernel");
    std::printf("H=%d, W=%d, K=%d: CPU/direct GPU PASS; CPU/shared GPU PASS\n",
                height, width, k);

    if (print_matrices) {
        std::printf("Input image:\n");
        for (int y = 0; y < height; ++y) {
            for (int x = 0; x < width; ++x)
                std::printf("%7.3f", h_image[y * width + x]);
            std::printf("\n");
        }
        std::printf("CPU reference / direct GPU / shared GPU:\n");
        for (int y = 0; y < height; ++y) {
            for (int x = 0; x < width; ++x) {
                const int i = y * width + x;
                std::printf("%6.3f/%6.3f/%6.3f  ",
                            h_reference[i], h_direct[i], h_shared[i]);
            }
            std::printf("\n");
        }
    }

    if (measure_time) {
        for (int warmup = 0; warmup < 5; ++warmup) {
            launch_direct(d_image, d_direct, height, width, k);
            launch_shared(d_image, d_shared, height, width, k);
        }
        CUDA_CHECK(cudaDeviceSynchronize());
        TimingGPU timer;
        float direct_ms[BATCHES] = {};
        float shared_ms[BATCHES] = {};
        for (int batch = 0; batch < BATCHES; ++batch) {
            const bool shared_first = (batch % 2) != 0;
            for (int position = 0; position < 2; ++position) {
                const bool use_shared = shared_first ? (position == 0) : (position == 1);
                timer.StartCounter();
                for (int repeat = 0; repeat < REPEATS; ++repeat) {
                    if (use_shared)
                        launch_shared(d_image, d_shared, height, width, k);
                    else
                        launch_direct(d_image, d_direct, height, width, k);
                }
                const float ms = timer.GetCounter() / REPEATS;
                if (use_shared) shared_ms[batch] = ms;
                else            direct_ms[batch] = ms;
            }
        }
        float direct_mean = 0.0f, shared_mean = 0.0f;
        for (int batch = 0; batch < BATCHES; ++batch) {
            direct_mean += direct_ms[batch];
            shared_mean += shared_ms[batch];
        }
        direct_mean /= BATCHES;
        shared_mean /= BATCHES;
        std::printf("Mean of %d batches x %d launches; kernel order alternated.\n",
                    BATCHES, REPEATS);
        std::printf("Direct: %.6f ms/launch; shared: %.6f ms/launch; direct/shared: %.3f\n",
                    direct_mean, shared_mean, direct_mean / shared_mean);
    }

    CUDA_CHECK(cudaFree(d_image));
    CUDA_CHECK(cudaFree(d_direct));
    CUDA_CHECK(cudaFree(d_shared));
    std::free(h_image); std::free(h_reference);
    std::free(h_direct); std::free(h_shared);
}

int main() {
    // Small non-divisible image: checks partial blocks and boundary handling.
    run_case(5, 7, 5, true, false);

    // Larger square image: compare narrow and wider 2D filters.
    const int height = 1024;
    const int width = 1024;
    const int filter_widths[] = {3, 7, 15};
    for (int k : filter_widths)
        run_case(height, width, k, false, true);
    return EXIT_SUCCESS;
}


Writing code/chapter02_2d_box_filter.cu


## Compile and run

The small `5×7` image is not a multiple of the `32×8` block dimensions, so it also exercises partial blocks at the image edges. The larger runs use a `1024×1024` image and filter widths `K=3, 7, 15`. Each run checks both GPU outputs against the CPU, then reports the mean of five timing batches with 20 launches each and alternated kernel order. A `direct/shared` ratio above 1 means the shared-memory kernel was faster on the selected GPU; no speedup is assumed.

In [6]:
%%bash
set -eu
nvcc -O2 -std=c++14 -Icode code/chapter02_2d_box_filter.cu code/TimingGPU.cu -o box_filter_2d
./box_filter_2d


H=5, W=7, K=5: CPU/direct GPU PASS; CPU/shared GPU PASS
Input image:
  0.000  0.136  0.273  0.409  0.545  0.682  0.818
  0.318  0.455  0.591  0.727  0.864  1.000  0.091
  0.636  0.773  0.909  0.000  0.136  0.273  0.409
  0.955  0.045  0.182  0.318  0.455  0.591  0.727
  0.227  0.364  0.500  0.636  0.773  0.909  0.000
CPU reference / direct GPU / shared GPU:
 0.164/ 0.164/ 0.164   0.209/ 0.209/ 0.209   0.271/ 0.271/ 0.271   0.311/ 0.311/ 0.311   0.309/ 0.309/ 0.309   0.238/ 0.238/ 0.238   0.193/ 0.193/ 0.193  
 0.211/ 0.211/ 0.211   0.269/ 0.269/ 0.269   0.349/ 0.349/ 0.349   0.375/ 0.375/ 0.375   0.400/ 0.400/ 0.400   0.322/ 0.322/ 0.322   0.264/ 0.264/ 0.264  
 0.255/ 0.255/ 0.255   0.338/ 0.338/ 0.338   0.449/ 0.449/ 0.449   0.502/ 0.502/ 0.502   0.513/ 0.513/ 0.513   0.415/ 0.415/ 0.415   0.331/ 0.331/ 0.331  
 0.238/ 0.238/ 0.238   0.305/ 0.305/ 0.305   0.395/ 0.395/ 0.395   0.420/ 0.420/ 0.420   0.404/ 0.404/ 0.404   0.316/ 0.316/ 0.316   0.249/ 0.249/ 0.249  
 0.184/ 0.184/ 0.184